In [38]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage,HumanMessage
from langchain_core.prompts import SystemMessagePromptTemplate,HumanMessagePromptTemplate,ChatPromptTemplate,ChatMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pathlib import Path
import tiktoken

In [39]:
load_dotenv(override=True)
MODEL = "qwen3:8b"
base_url = "http://localhost:11434"
llm = ChatOllama(base_url=base_url,model=MODEL)
llm_groq = ChatGroq(temperature=0,model="openai/gpt-oss-120b",api_key=os.getenv("GROQ_API_KEY"))
openrouter_llm = ChatOpenAI(
    model="qwen/qwen3.8-27b",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)
parser = StrOutputParser()

In [40]:
base_path = Path("../../rag-dataset")
pdfs = [str(p) for p in base_path.rglob("*.pdf")]
pdfs

['..\\..\\rag-dataset\\gym supplements\\1. Analysis of Actual Fitness Supplement.pdf',
 '..\\..\\rag-dataset\\gym supplements\\2. High Prevalence of Supplement Intake.pdf',
 '..\\..\\rag-dataset\\health supplements\\1. dietary supplements - for whom.pdf',
 '..\\..\\rag-dataset\\health supplements\\2. Nutraceuticals research.pdf',
 '..\\..\\rag-dataset\\health supplements\\3.health_supplements_side_effects.pdf']

In [41]:
docs = []
for pdf in pdfs:
    loder = PyMuPDFLoader(pdf)
    temp = loder.load()
    docs.extend(temp)  

In [42]:
def format_docs(docs):
    return "\n\n".join([x.page_content for x in docs]) 

In [43]:
context = format_docs(docs)
# print(context)

In [44]:
# system_message = SystemMessage("""You are helpful AI assistant who answer user question based on the provided context. 
#                                   Do not answer in more than {words} words""")
# user_message = HumanMessage("""Answer user question based on the provided context ONLY! 
#                               If you do not know the answer, just say "I don't know".
#             ### Context:
#             {context}

#             ### Question:
#             {question}

#             ### Answer:""")

In [45]:
system_message = SystemMessage("""You are helpful AI assistant who works as document summarizer. 
                                                   You must not hallucinate or provide any false information.""")
user_message = HumanMessage("""Summarize the given context in {words}.
            ### Context:
            {context}

            ### Summary:""")

In [46]:
system_prompt = SystemMessagePromptTemplate.from_template(system_message.content)
user_prompt = HumanMessagePromptTemplate.from_template(user_message.content)

messages = [system_prompt,user_prompt]

template = ChatPromptTemplate.from_messages(messages)

chain = template | openrouter_llm | parser



output = chain.invoke({'context': context, 'words': 500})
print(output)



Across the provided studies, supplement use is common among fitness participants, but it is shaped by motivation, information quality, regulation, and safety concerns. A 2024 Philippine study of 250 fitness supplement consumers integrated the Theory of Planned Behavior, Health Belief Model, and Theory of Effort Minimization in Physical Activity to examine actual consumption. Structural equation modeling showed that health motivation, attitude, perceived behavioral control, and controlled precursors significantly influenced behavioral intention, while behavioral intention strongly predicted actual consumption. In contrast, subjective norms, perceived effectiveness, perceived benefits, perceived barriers, and automatic precursors were not significant. The authors interpret this as evidence that consumers act largely from personal health goals, confidence, sense of control, and deliberate evaluation rather than social pressure, assumed effectiveness, or habitual cues. They recommend mar

In [47]:
response = chain.invoke({'context': context, 
                             'question': "Provide a detailed report from the provided context. Write answer in Markdown languge only example .md .", 
                             'words': 2000})
print(response)



The provided context presents a collection of studies and reviews concerning supplement and nutraceutical use, the factors that drive consumption, the quality of information users receive, the scientific evidence on health effects, safety risks, and regulatory gaps. The materials include a Philippine behavioral study, a Swiss prevalence survey, reviews of dietary supplement effects on chronic disease, and discussions of nutraceutical regulation and adverse effects. Together, they show that supplement consumption is widespread among fitness users and general populations, often motivated by health, performance, aesthetics, and perceived convenience, but it is not always accompanied by reliable knowledge, clinical evidence, or regulatory assurance.

The first article by Espeño et al. investigates actual fitness supplement consumption among health and fitness enthusiasts in the Philippines. The authors integrate the Theory of Planned Behavior, the Health Belief Model, and the Theory of E